In [1]:
import pandas as pd

tickers_list = ['NIO','AMC','DAWN','PINS','TSLA']

probs_list = [0.9,0.85,0.8,0.7,0.5]
probs_di = dict(zip(tickers_list,probs_list))

perc_list = [6,8,10,12,4] # assume up and down risks are the same
perc_di = dict(zip(tickers_list,perc_list))

prices_list = [21.12,14.15,8.47,40.34,745.00]
prices_di = dict(zip(tickers_list,prices_list))

cash = 50_000
# how to generate a ratio list that sums up to 1
equity_list = [10_000,10_000,10_000,10_000,10_000]
equity_di = dict(zip(tickers_list,equity_list))

expected_pnl = 0
for ticker in tickers_list:
    expected_pnl += equity_di[ticker]*perc_di[ticker]/100*(probs_di[ticker]-(1-probs_di[ticker]))
print(expected_pnl)

2120.0


In [14]:
import re
import itertools
scenarios = [f'{i}fail' for i in range(len(tickers_list)+1)]
scenario_probs = {}
scenario_pnl = {}
for scenario in scenarios:
    fail_num = [int(s) for s in re.findall(r'\d+',scenario)][0]
    if fail_num == 0:
        prob = 1
        earn = 0
        for ticker in tickers_list:
            prob *= probs_di[ticker]
            earn += equity_di[ticker]*perc_di[ticker]/100
        scenario_probs[scenario] = prob
        scenario_pnl[scenario] = earn
    elif fail_num == len(tickers_list):
        prob = 1
        earn = 0
        for ticker in tickers_list:
            prob *= (1-probs_di[ticker])
            earn -= equity_di[ticker]*perc_di[ticker]/100
        scenario_probs[scenario] = prob
        scenario_pnl[scenario] = earn        
    else:
        failed_tickers_li = list(itertools.combinations(tickers_list,fail_num))
        prob_li = []
        earn_li = []
        for failed_tickers in failed_tickers_li:
            won_tickers = []
            [won_tickers.append(x) for x in tickers_list if x not in failed_tickers]
            prob = 1
            earn = 0
            for ticker in failed_tickers:
                prob *= (1-probs_di[ticker])
                earn -= equity_di[ticker]*perc_di[ticker]/100
            for ticker in won_tickers:
                prob *= probs_di[ticker]
                earn += equity_di[ticker]*perc_di[ticker]/100
            prob_li.append(prob)
            earn_li.append(earn)
        scenario_probs[scenario] = sum(prob_li)
        norm_prob_li = [x/sum(prob_li) for x in prob_li]
        scenario_pnl[scenario] = sum([a*b for a,b in zip(norm_prob_li,earn_li)])

df = pd.DataFrame(index=scenarios)
for ind in df.index:
    df.loc[ind,['pnl','prob']] = scenario_pnl[ind],scenario_probs[ind]
df['cum_prob'] = df['prob'].cumsum()
df



,pnl,prob,cum_prob
0fail,4000.000000,0.21420,0.21420
1fail,2604.250267,0.42115,0.63535
2fail,921.565785,0.27590,0.91125
3fail,-789.030612,0.07840,0.98965
4fail,-2442.424242,0.00990,0.99955
5fail,-4000.000000,0.00045,1.00000


In [3]:
# verify if the calculations are correct
expected_pnl = 0
for k in scenario_probs.keys():
    expected_pnl += scenario_probs[k]*scenario_pnl[k]
print(expected_pnl)


2120.0000000000005


In [12]:
df

,pnl,prob,cum_prob
0fail,4000.000000,0.21420,0.21420
1fail,2604.250267,0.42115,0.63535
2fail,921.565785,0.27590,0.91125
3fail,-789.030612,0.07840,0.98965
4fail,-2442.424242,0.00990,0.99955
5fail,-4000.000000,0.00045,1.00000
